To make this model work, we need to do the following: 
* For all crashes in our study area (Oakland + Berkeley), we need to mark whether a crash is associated with a traffic island and signal
* We need to determine which other explanatory variables to include in the model:
    * Traffic signal presence (OSM)
    * Traffic refuge island presence (OSM)
    * Number of lanes (TBD)
        * Looks like you have to play with the overpass API to get this data from OSM https://stackoverflow.com/questions/56558717/query-all-roads-with-overpass-api-and-export-as-polygon
    * Ped characteristics (gender, age, etc.) (retrieve from SWITRS)
        * **How do we do this if ped characteristics are tied to crash party but we want to look at crashes overall?**
    * AADT (From replica)
    * Functional classification of road as a proxy for speed
    * Day of the week (SWITRS)
    * Lighting at intersection (TBD, possibly on OSM)

In [192]:
import pandas as pd
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel

In [193]:
# Load dataset
file_path = "crash_data_for_model.csv"  # Update with the actual file path
df = pd.read_csv(file_path, dtype=str)

In [194]:
# Remove irrelevant columns

df = df[[
        'COLLISION_SEVERITY',
    'AT_FAULT', # No should be reference
    'PARTY_SEX', # Male should be reference
    'PARTY_AGE',
    'RACE', # White should be reference
    'LIGHTING', # Daylight should be reference
    'DAY_OF_WEEK',
    'WEATHER_1', # Clear should be reference
    'PRIMARY_COLL_FACTOR', # Make dummy for A - Vehicle Code Violation
    'PED_ACTION', # B - Crossing in Crosswalk at Intersection should be reference
    'PARTY_NUMBER_KILLED',
    'PARTY_NUMBER_INJURED',
    'road_class',
    'island_id', # Convert to 1/0
    'volume', 
    'signal_id' # Convert to 1/0
]]

In [195]:
def remap_severity(severity_series):
    # Convert severity values to numeric, coercing errors to NaN
    severity_numeric = pd.to_numeric(severity_series, errors="coerce")
    # Define the mapping from original to reversed scale
    severity_mapping = {1: 4, 2: 3, 3: 2, 4: 1}
    remapped = severity_numeric.map(severity_mapping)
    # Define the order for the categorical variable
    severity_order = [1, 2, 3, 4]
    return pd.Categorical(remapped, categories=severity_order, ordered=True)

# Apply severity remapping using the dedicated function
df["COLLISION_SEVERITY"] = remap_severity(df["COLLISION_SEVERITY"])

In [196]:
# # Collapse DAY_OF_WEEK into "Weekday" (Mon-Fri) vs. "Weekend" (Sat-Sun)
# df["DAY_OF_WEEK"] = df["DAY_OF_WEEK"].apply(lambda x: "Weekday" if x in ["1", "2", "3", "4", "5"] else "Weekend")

In [197]:
# Remap at fault category
fault_dict = {'N':0, 'Y':1}
df['AT_FAULT'] = df['AT_FAULT'].map(fault_dict)
df['AT_FAULT'] = df['AT_FAULT']#.astype(int)
df['AT_FAULT'].value_counts()

AT_FAULT
0    610
1     60
Name: count, dtype: int64

In [198]:
# Remap island data
print(df['island_id'].value_counts().sum())
df['island_id'] = df['island_id'].notna()#.astype(int)
df['island_id'] = df['island_id'].map(int)
print(df['island_id'].value_counts())


36
island_id
0    634
1     36
Name: count, dtype: int64


In [199]:
# Remap signal data
print(df['signal_id'].value_counts().sum())
df['signal_id'] = df['signal_id'].notna()#.astype(int)
df['signal_id'] = df['signal_id'].map(int)
print(df['signal_id'].value_counts())

169
signal_id
0    501
1    169
Name: count, dtype: int64


In [200]:
# Apply dummy encoding, keeping "Weekday" as the reference category
df = pd.get_dummies(df, columns=[
    'PARTY_SEX', # Male should be reference
    'RACE', # White should be reference
    'LIGHTING', # Daylight should be reference
    'WEATHER_1', # Clear should be reference
    'PRIMARY_COLL_FACTOR', # Make dummy for A - Vehicle Code Violation
    'PED_ACTION', # B - Crossing in Crosswalk at Intersection should be reference
    ], drop_first=True)

display(df)

,COLLISION_SEVERITY,AT_FAULT,PARTY_AGE,DAY_OF_WEEK,PARTY_NUMBER_KILLED,PARTY_NUMBER_INJURED,road_class,island_id,volume,signal_id,...,PRIMARY_COLL_FACTOR_A,PRIMARY_COLL_FACTOR_B,PRIMARY_COLL_FACTOR_C,PRIMARY_COLL_FACTOR_D,PED_ACTION_B,PED_ACTION_C,PED_ACTION_D,PED_ACTION_E,PED_ACTION_F,PED_ACTION_G
0,1,0,60,2,0,1,4,0,188606.62996098422,0,...,True,False,False,False,True,False,False,False,False,False
1,1,0,57,1,0,1,5,0,1183902.7845568222,0,...,True,False,False,False,True,False,False,False,False,False
2,2,0,30,5,0,1,5,0,1183902.7845568222,0,...,True,False,False,False,True,False,False,False,False,False
3,2,1,13,7,0,1,5,0,1183902.7845568222,0,...,True,False,False,False,True,False,False,False,False,False
4,1,0,23,3,0,1,7,0,3262.780625864375,0,...,True,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,2,0,45,4,0,1,4,0,605870.8697948474,0,...,True,False,False,False,True,False,False,False,False,False
666,3,0,77,2,0,1,5,0,33858.13232104739,0,...,True,False,False,False,True,False,False,False,False,False
667,2,0,42,6,0,1,4,0,266336.5801759236,0,...,True,False,False,False,True,False,False,False,False,False
668,1,0,38,4,0,1,4,0,560856.5575155583,0,...,True,False,False,False,True,False,False,False,False,False


In [201]:
# Fix all the data types

df.dtypes

COLLISION_SEVERITY       category
AT_FAULT                    int64
PARTY_AGE                  object
DAY_OF_WEEK                object
PARTY_NUMBER_KILLED        object
PARTY_NUMBER_INJURED       object
road_class                 object
island_id                   int64
volume                     object
signal_id                   int64
PARTY_SEX_F                  bool
PARTY_SEX_M                  bool
PARTY_SEX_X                  bool
RACE_B                       bool
RACE_H                       bool
RACE_O                       bool
RACE_W                       bool
LIGHTING_A                   bool
LIGHTING_B                   bool
LIGHTING_C                   bool
LIGHTING_D                   bool
LIGHTING_E                   bool
WEATHER_1_A                  bool
WEATHER_1_B                  bool
WEATHER_1_C                  bool
WEATHER_1_E                  bool
WEATHER_1_F                  bool
PRIMARY_COLL_FACTOR_A        bool
PRIMARY_COLL_FACTOR_B        bool
PRIMARY_COLL_F

In [202]:
df['volume']

0      188606.62996098422
1      1183902.7845568222
2      1183902.7845568222
3      1183902.7845568222
4       3262.780625864375
              ...        
665     605870.8697948474
666     33858.13232104739
667     266336.5801759236
668     560856.5575155583
669     8842.145632099371
Name: volume, Length: 670, dtype: object

In [203]:
cols = ['PARTY_AGE', 'DAY_OF_WEEK', 'PARTY_NUMBER_KILLED', 'PARTY_NUMBER_INJURED', 'road_class']

df[cols] = df[cols].astype(int)

df['volume'] = df['volume'].astype(float)
df['volume'] = df['volume'].fillna(0).astype(int)
# lets scale the volumes between 0 and 1 so they aren't such ridiculous numeric outliers
max_value = df['volume'].max()
df['volume'] = df['volume']/max_value
# ^ we could consider binning these instead

# let's convert all the True/False into 1/0
for col in df:
    if df[col].dtype == bool:
        df[col] = df[col].map(int)

df.dtypes

COLLISION_SEVERITY       category
AT_FAULT                    int64
PARTY_AGE                   int64
DAY_OF_WEEK                 int64
PARTY_NUMBER_KILLED         int64
PARTY_NUMBER_INJURED        int64
road_class                  int64
island_id                   int64
volume                    float64
signal_id                   int64
PARTY_SEX_F                 int64
PARTY_SEX_M                 int64
PARTY_SEX_X                 int64
RACE_B                      int64
RACE_H                      int64
RACE_O                      int64
RACE_W                      int64
LIGHTING_A                  int64
LIGHTING_B                  int64
LIGHTING_C                  int64
LIGHTING_D                  int64
LIGHTING_E                  int64
WEATHER_1_A                 int64
WEATHER_1_B                 int64
WEATHER_1_C                 int64
WEATHER_1_E                 int64
WEATHER_1_F                 int64
PRIMARY_COLL_FACTOR_A       int64
PRIMARY_COLL_FACTOR_B       int64
PRIMARY_COLL_F

In [204]:
# Define independent variables (all columns except the target)
independent_vars = df.columns.difference(["COLLISION_SEVERITY"])
independent_vars

Index(['AT_FAULT', 'DAY_OF_WEEK', 'LIGHTING_A', 'LIGHTING_B', 'LIGHTING_C',
       'LIGHTING_D', 'LIGHTING_E', 'PARTY_AGE', 'PARTY_NUMBER_INJURED',
       'PARTY_NUMBER_KILLED', 'PARTY_SEX_F', 'PARTY_SEX_M', 'PARTY_SEX_X',
       'PED_ACTION_B', 'PED_ACTION_C', 'PED_ACTION_D', 'PED_ACTION_E',
       'PED_ACTION_F', 'PED_ACTION_G', 'PRIMARY_COLL_FACTOR_A',
       'PRIMARY_COLL_FACTOR_B', 'PRIMARY_COLL_FACTOR_C',
       'PRIMARY_COLL_FACTOR_D', 'RACE_B', 'RACE_H', 'RACE_O', 'RACE_W',
       'WEATHER_1_A', 'WEATHER_1_B', 'WEATHER_1_C', 'WEATHER_1_E',
       'WEATHER_1_F', 'island_id', 'road_class', 'signal_id', 'volume'],
      dtype='object')

In [207]:
# Specify and calibrate the Ordered Logit Model
model = OrderedModel(
    endog=df["COLLISION_SEVERITY"],
    exog=df[independent_vars],
    distr="logit"
)

# I tried some different model methods. I think we may have too many un-useful independent variables
result = model.fit(method="cg")
# result = model.fit(method="nm")
# result = model.fit(method="bfgs")
print(result.summary())

/opt/anaconda3/lib/python3.13/site-packages/scipy/optimize/_optimize.py:1680: OptimizeWarning: Maximum number of iterations has been exceeded.
  res = _minimize_cg(f, x0, args, fprime, callback=callback, c1=c1, c2=c2,


         Current function value: 0.934255
         Iterations: 500
         Function evaluations: 1245
         Gradient evaluations: 1236
                             OrderedModel Results                             
Dep. Variable:     COLLISION_SEVERITY   Log-Likelihood:                -625.95
Model:                   OrderedModel   AIC:                             1330.
Method:            Maximum Likelihood   BIC:                             1506.
Date:                Sun, 19 Apr 2026                                         
Time:                        13:12:14                                         
No. Observations:                 670                                         
Df Residuals:                     631                                         
Df Model:                          36                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [206]:
# Compute and display Odds Ratios
odds_ratios = np.exp(result.params)
print("\nOdds Ratios:\n", odds_ratios)

# Predict probabilities for each severity level
predicted_probs = result.predict()

# Convert predictions to a DataFrame
predicted_probs_df = pd.DataFrame(predicted_probs)

# Determine the most likely severity level for each observation
df["Predicted_Severity"] = predicted_probs_df.idxmax(axis=1)

# Display performance metrics with a confusion matrix (objective results only)
confusion_matrix = pd.crosstab(df["COLLISION_SEVERITY"], df["Predicted_Severity"],
                               rownames=['Actual'], colnames=['Predicted'])
print("\nConfusion Matrix:\n", confusion_matrix)



Odds Ratios:
 AT_FAULT                 1.428278e+00
DAY_OF_WEEK              9.484503e-01
LIGHTING_A               4.201126e-01
LIGHTING_B               2.621059e-01
LIGHTING_C               4.689629e-01
LIGHTING_D               6.824376e-01
LIGHTING_E               3.347056e+00
PARTY_AGE                1.000421e+00
PARTY_NUMBER_INJURED     2.587755e-01
PARTY_NUMBER_KILLED      1.262545e+17
PARTY_SEX_F              5.780152e-01
PARTY_SEX_M              6.187009e-01
PARTY_SEX_X              2.350321e-08
PED_ACTION_B             3.007364e+05
PED_ACTION_C             8.633584e+04
PED_ACTION_D             5.738874e+05
PED_ACTION_E             2.947119e+05
PED_ACTION_F             1.614057e+05
PED_ACTION_G             9.548080e-03
PRIMARY_COLL_FACTOR_A    6.862154e-01
PRIMARY_COLL_FACTOR_B    2.298489e+00
PRIMARY_COLL_FACTOR_C    6.614289e+00
PRIMARY_COLL_FACTOR_D    3.645154e-01
RACE_B                   6.500591e-01
RACE_H                   6.953313e-01
RACE_O                   1.574001e+

In [157]:
display(df)

,COLLISION_SEVERITY,AT_FAULT,PARTY_AGE,DAY_OF_WEEK,PARTY_NUMBER_KILLED,PARTY_NUMBER_INJURED,road_class,island_id,volume,signal_id,...,PRIMARY_COLL_FACTOR_B,PRIMARY_COLL_FACTOR_C,PRIMARY_COLL_FACTOR_D,PED_ACTION_B,PED_ACTION_C,PED_ACTION_D,PED_ACTION_E,PED_ACTION_F,PED_ACTION_G,Predicted_Severity
0,1,0,60,2,0,1,4,0,188606,0,...,0,0,0,1,0,0,0,0,0,1
1,1,0,57,1,0,1,5,0,1183902,0,...,0,0,0,1,0,0,0,0,0,1
2,2,0,30,5,0,1,5,0,1183902,0,...,0,0,0,1,0,0,0,0,0,1
3,2,1,13,7,0,1,5,0,1183902,0,...,0,0,0,1,0,0,0,0,0,1
4,1,0,23,3,0,1,7,0,3262,0,...,0,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,2,0,45,4,0,1,4,0,605870,0,...,0,0,0,1,0,0,0,0,0,1
666,3,0,77,2,0,1,5,0,33858,0,...,0,0,0,1,0,0,0,0,0,1
667,2,0,42,6,0,1,4,0,266336,0,...,0,0,0,1,0,0,0,0,0,1
668,1,0,38,4,0,1,4,0,560856,0,...,0,0,0,1,0,0,0,0,0,1
